In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy preprocessing.py
COPY preprocessing.py .

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
scikit_learn==0.24.1
boto3==1.24.59
catboost==1.0.4

Writing requirements.txt


### Write ```preprocessing.py```

In [4]:
%%writefile preprocessing.py

from sklearn.base import BaseEstimator, TransformerMixin
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
import boto3

# replace nana
class ReplaceNaNs(BaseEstimator, TransformerMixin):
    # init
    def __init__(self, bool_verbose=True, str_message='NaN replacer'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # replace NaNs
        time_start = time.perf_counter()

        X.replace(['None', None, 'NaN', 'nan', ''], np.nan, inplace=True)

        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

# replace booleans
class ReplaceBooleans(BaseEstimator, TransformerMixin):
    # init
    def __init__(self, bool_verbose=True, str_message='Boolean replacer'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # replace NaNs
        time_start = time.perf_counter()
        
        # replacement dictionary
        dict_replace = {
            True: 1,
            False: 0,
            'True': 1,
            'False': 0,
            '0': 0,
            '1': 1,
        }
        X.replace(dict_replace, inplace=True)

        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

# data type setter
class SetDataTypes(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, bool_verbose=True, str_message='Data Type Setter'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        # get the data types into a dictionary
        dict_dtypes = dict(X.dtypes)
        # save to object
        self.dict_dtypes = dict_dtypes
        return self
    # transform
    def transform(self, X):
        # fillna
        time_start = time.perf_counter()

        # rm key val pairs not in X
        dict_dtypes = {key: val for key, val in self.dict_dtypes.items() if key in list(X.columns)}
    
        # change O to str
        dict_dtypes = {key: ('str' if val == 'O' else val) for key, val in dict_dtypes.items()}
        # iterate
        for key, val in tqdm (dict_dtypes.items()):
            X[key] = X[key].astype(val)

        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

# class for cleaning text
class CleanText(BaseEstimator, TransformerMixin):
    # initialize class
    def __init__(self, list_cols, bool_verbose=True, str_message='Text Cleaner'):
        self.list_cols = list_cols
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # start timer
        time_start = time.perf_counter()

        # future proof
        list_cols = [col for col in self.list_cols if col in list(X.columns)]
        
        # iterate
        for col in tqdm (list_cols):
            # convert to string
            X[col] = X[col].astype(str)
            # lower, strip, replace
            X[col] = X[col].str.lower().str.strip().str.replace(' ', '')
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # return
        return X

# class for inflation
class Inflator(BaseEstimator, TransformerMixin):
    # initialize class
    def __init__(self, list_cols, dict_inflation_rate, bool_verbose=True, str_datecol='applicationdate__app', str_message='Inflator'):
        self.list_cols = list_cols
        self.dict_inflation_rate = dict_inflation_rate
        self.bool_verbose = bool_verbose
        self.str_datecol = str_datecol
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # start timer
        time_start = time.perf_counter()

        # future proof
        list_cols = [col for col in self.list_cols if col in list(X.columns)]
        
        # create year
        X['year'] = X[self.str_datecol].dt.year
        # map factor to year
        X['factor'] = X['year'].map(self.dict_inflation_rate)

        # convert
        for col in tqdm (list_cols):
            # multiply by factor
            X[col] = X[col] * X['factor']

        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # return
        return X

# clip values
class ClipValues(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, list_cols, bool_verbose=True, str_message='Value clipper', a_min=0, a_max=None):
        self.list_cols = list_cols
        self.bool_verbose = bool_verbose
        self.str_message = str_message
        self.a_min = a_min
        self.a_max = a_max
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # start time
        time_start = time.perf_counter()

        # future proof
        list_cols = [col for col in self.list_cols if col in list(X.columns)]

        # if iterating
        for col in tqdm (list_cols):
            X[col] = np.clip(a=X[col], a_min=self.a_min, a_max=self.a_max)
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return X
        return X

# custom imputer
class CustomImputer(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, dict_imputation, bool_verbose=True, str_message='Imputer'):
        self.dict_imputation = dict_imputation
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        # return object
        return self
    # transform
    def transform(self, X):
        # fillna
        time_start = time.perf_counter()
        
        # future proof
        list_cols = list(self.dict_imputation.keys())
        list_cols = [col for col in list_cols if col in list(X.columns)]
        
        # impute
        for col in tqdm (list_cols):
            X[col] = X[col].fillna(self.dict_imputation[col])
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X
    
# imputer
class Imputer(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, bool_verbose=True, str_message='Imputer'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        # return object
        return self
    # transform
    def transform(self, X):
        # fillna
        time_start = time.perf_counter()
        
        # impute
        X = X.fillna(0)

        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

# define value replacer class
class FeatureValueReplacer(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, dict_value_replace, bool_verbose=True, str_message='Value Replacer'):
        self.dict_value_replace = dict_value_replace
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # start time
        time_start = time.perf_counter()

        # future proof
        dict_value_replace = {key: val for key, val in self.dict_value_replace.items() if key in list(X.columns)}
        # replace
        for key, val in tqdm (dict_value_replace.items()):
            X[key] = X[key].replace(val)

        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

# date features
class DateFeatures(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, bool_verbose=True, str_message='Date features'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # fillna
        time_start = time.perf_counter()
        
        # make sure date is datetime
        X['applicationdate__app'] = pd.to_datetime(X['applicationdate__app'])
        
        # date features
        X['ENG-applicationdate__app_month'] = X['applicationdate__app'].dt.month
        X['ENG-applicationdate__app_quarter'] = X['applicationdate__app'].dt.quarter

        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

# rounding binner
class RoundBinning(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, dict_round, bool_verbose=True, str_message='Binner'):
        self.dict_round = dict_round
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X):
        return self
    # transform
    def transform(self, X):
        # start time
        time_start = time.perf_counter()

        # rm key val pairs not in X
        dict_round = {key: val for key, val in self.dict_round.items() if key in list(X.columns)}
        for key, val in tqdm (dict_round.items()):
            X[key] = val * round(pd.to_numeric(X[key]) / val)
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return X
        return X

# feature engineering
class FeatureEngineering(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, bool_verbose=True, str_message='Engineer PTI and LTV'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # fillna
        time_start = time.perf_counter()
        
        # payment to income
        try:
            X['ENG-payment_to_income'] = X['fltapprovedpayment__app'] / X['fltgrossmonthly__income_sum']
        except KeyError as e:
            print(f'Unable to engineer PTI: {e} is not in the data frame')
        # loan to value
        try:
            X['ENG-loan_to_value'] = X['fltamountfinanced__app'] / X['fltapprovedpricewholesale__app']
        except KeyError as e:
            print(f'Unable to engineer LTV: {e} is not found in the data frame')
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X 

# replace inf and -inf with NaN
class ReplaceInf(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, bool_verbose=True, str_message='Inf Replacer'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        # get list columns
        list_cols = list(X.columns)
        # save
        self.list_cols = list_cols
        # return object
        return self
    # transform
    def transform(self, X):
        # start time
        time_start = time.perf_counter()

        # future proof
        list_cols = [col for col in self.list_cols if col in list(X.columns)]

        # if find and replace
        for col in tqdm (list_cols):
            X[col] = X[col].replace([np.inf, -np.inf], np.nan)
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return X
        return X

# define function for mapping term
def custom_mapping_term(int_term):
    if int_term == 0:
        return 72
    elif int_term <= 12:
        return 12
    elif int_term <= 24:
        return 24
    elif int_term <= 36:
        return 36
    elif int_term <= 48:
        return 48
    elif int_term <= 60:
        return 60
    else:
        return 72

# map term
class MapTerm(BaseEstimator, TransformerMixin):
    # init
    def __init__(self, bool_verbose=True, str_message='Map term'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # map
        time_start = time.perf_counter()
        
        # map
        try:
            X['intterm__app'] = X['intterm__app'].apply(custom_mapping_term)
        except KeyError:
            pass
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X
        
# define function for mapping PTI
def custom_mapping_pti(flt_pti):
    if flt_pti <= 0:
        return 0.15
    elif flt_pti <= 0.03:
        return 0
    elif flt_pti <= 0.06:
        return 0.03
    elif flt_pti <= 0.09:
        return 0.06
    elif flt_pti <= 0.12:
        return 0.09
    elif flt_pti <= 0.15:
        return 0.12
    else:
        return 0.15

# map term
class MapPTI(BaseEstimator, TransformerMixin):
    # init
    def __init__(self, bool_verbose=True, str_message='Map PTI'):
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # map
        time_start = time.perf_counter()
        
        # map
        try:
            X['ENG-payment_to_income'] = X['ENG-payment_to_income'].apply(custom_mapping_pti)
        except KeyError:
            pass
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

# rounding binner
class RoundBinning(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, dict_round, bool_verbose=True, str_message='Binner'):
        self.dict_round = dict_round
        self.bool_verbose = bool_verbose
        self.str_message = str_message
    # fit
    def fit(self, X):
        return self
    # transform
    def transform(self, X):
        # start time
        time_start = time.perf_counter()

        # rm key val pairs not in X
        dict_round = {key: val for key, val in self.dict_round.items() if key in list(X.columns)}
        for key, val in tqdm (dict_round.items()):
            X[key] = val * round(pd.to_numeric(X[key]) / val)
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        if self.bool_verbose:
            print(f'{self.str_message}: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return X
        return X

# define preprocessing model class
class PreprocessingModel(BaseEstimator, TransformerMixin):
    # initialize
    def __init__(self, list_transformers):
        self.list_transformers = list_transformers
    # fit
    def fit(self, X, y=None):
        return self
    # transform
    def transform(self, X):
        # start time
        time_start = time.perf_counter()

        # iterate through transformers
        for transformer in self.list_transformers:
            X = transformer.transform(X)
        
        # end time
        time_end = time.perf_counter()
        # flt_sec
        flt_sec = time_end - time_start
        # print
        print(f'Preprocessing Model: {flt_sec:0.5} sec.')
        # save to object
        self.flt_sec = flt_sec
        # return
        return X

Writing preprocessing.py


### Write ```script.py``` to local drive

In [5]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import preprocessing as pre
import pickle
import boto3
from pandas.api.types import is_numeric_dtype

# constants
str_project = '20231010-gen-xii'
str_target = 'target'
str_datecol = 'applicationdate__app'
str_dirname_output = './output'

# show transformers
def show_transformers(list_transformers):
    # iterate
    for a, transformer in enumerate(list_transformers):
        print(f'{a+1}: {transformer.__class__.__name__} - {transformer.str_message}')

# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# error in case data still has missing vals
class DataContainsNaN(Exception):
    """Raise this when the data contains missing observations."""
    pass

# create output dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

# import training data
print('Importing training data...')
str_filename = 'df_train_raw.gzip'
str_uri = f's3://{str_project}/01_ad/01_data_prep/03_train_valid_test_split/{str_filename}'
df = pd.read_parquet(str_uri)

#####################################################################################
# CREATE EMPTY PIPELINE
#####################################################################################
print('Creating pipeline...')
list_transformers = []

#####################################################################################
# INITIALIZE TRANSFORMERS AND TRANSFORM DATA
#####################################################################################
# replace NaNs
print('Replacing NaNs...')
# init
cls_transformer = pre.ReplaceNaNs(
    bool_verbose=True, 
    str_message='NaN Replacer',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# replace Booleans
print('Replacing Booleans...')
# init
cls_transformer = pre.ReplaceBooleans(
    bool_verbose=True,
    str_message='Boolean Replacer',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# set dtypes
print('Setting dtypes...')
# hard code
df['inttype__app'] = df['inttype__app'].astype(str)
# init
cls_transformer = pre.SetDataTypes(
    bool_verbose=True, 
    str_message='Data Type Setter',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# clean text
print('Cleaning text...')
# get string cols
list_cols_nonnumeric = []
for col in df.columns:
    str_dtype = df[col].dtype
    # logic
    if str_dtype not in ['int64','float64']:
        list_cols_nonnumeric.append(col)
    else:
        pass
# make sure date col not converted to string
list_cols_nonnumeric = [col for col in list_cols_nonnumeric if col != str_datecol]
print(f'There are {len(list_cols_nonnumeric)} non-numeric features:')
#pprint(list_cols)
cls_transformer = pre.CleanText(
    list_cols=list_cols_nonnumeric, 
    bool_verbose=True,
    str_message='Clean text and impute non-numeric',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

######################################################################
print('Inflating automobile features...')
# hard code inflation rates from cpi
dict_inflation_rate_auto = {
    2013: 1.1343492088654628,
    2014: 1.135868486844716,
    2015: 1.136341981459793,
    2016: 1.1425221287509106,
    2017: 1.1579016565869051,
    2018: 1.1549581357812975,
    2019: 1.1512071028526036,
}
# read in df_dollars
str_filename = 'df_dollars.csv'
str_uri = f's3://{str_project}/01_ad/02_model/00_preprocessing/01_create_preprocessor/{str_filename}'
df_dollars = pd.read_csv(str_uri)
# get list of dollar features
list_cols_dollars_auto = list(df_dollars[df_dollars['auto_flag']==1]['feature'])
# hard code list of auto features
list_feats_auto = [
    'fltamountfinanced__app',
    'fltgapinsurance__app',
    'fltservicecontract__app',
    'fltapprovedpricewholesale__app',
    'fltapprovedpayment__app',
]
# make sure they are in list_cols_dollars_auto
list_cols_dollars_auto = list_cols_dollars_auto + list_feats_auto
# rm dups
list_cols_dollars_auto = list(dict.fromkeys(list_cols_dollars_auto))
# fit transform
cls_transformer = pre.Inflator(
    list_cols=list_cols_dollars_auto,
    dict_inflation_rate=dict_inflation_rate_auto, 
    str_datecol=str_datecol, 
    str_message='Inflate to 2021 dollars (automobiles only)', 
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

print('Inflating non-automobile features...')
# hard code inflation rates from cpi
dict_inflation_rate_non_auto = {
    2013: 1.16317603677932,
    2014: 1.144608340091917,
    2015: 1.143251327963817,
    2016: 1.1290087372451638,
    2017: 1.1054585509138384,
    2018: 1.079101737506322,
    2019: 1.059896658413421,    
}
# get list of dollar features
list_cols_dollars_non_auto = list(df_dollars[df_dollars['auto_flag']==0]['feature'])
# rm any auto features
list_cols_dollars_non_auto = [col for col in list_cols_dollars_non_auto if col not in list_cols_dollars_auto]
# fit transform
cls_transformer = pre.Inflator(
    list_cols=list_cols_dollars_non_auto,
    dict_inflation_rate=dict_inflation_rate_non_auto, 
    str_datecol=str_datecol, 
    str_message='Inflate to 2021 dollars (non-automobile)', 
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

print('Clipping dollar features...')
# get dollar features
list_cols_dollars = list_cols_dollars_non_auto + list_cols_dollars_auto
# fit transform
cls_transformer = pre.ClipValues(
    list_cols=list_cols_dollars, 
    bool_verbose=True, 
    str_message='Clip negative dollar values to zero (automobile and non-automobile)', 
    a_min=0,
    a_max=None,
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

print('Clipping income count...')
# fit transform
cls_transformer = pre.ClipValues(
    list_cols=['fltgrossmonthly__income_count'], 
    bool_verbose=True, 
    str_message='Clip number of income sources to 2', 
    a_min=0,
    a_max=2,
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

######################################################################
# custom imputation
print('Custom imputation...')
dict_imputation = {
    'intservicecontractmileage__app': 125000,
}
# init
cls_transformer = pre.CustomImputer(
    dict_imputation=dict_imputation,
    bool_verbose=True, 
    str_message='Custom imputer',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# impute
print('Imputing numeric with 0...')
# init
cls_transformer = pre.Imputer(
    bool_verbose=True, 
    str_message='Imputer',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# feature value replacer
print('Replacing 0s with predetermined values...')
dict_value_replace = {
    'fltamountfinanced__app': {0: 45000},
    'fltapprovedpricewholesale__app': {0: 28125},
    
}
cls_transformer = pre.FeatureValueReplacer(
    dict_value_replace=dict_value_replace, 
    str_message='Replace zeros with predetermined value and change ram to dodge',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

######################################################################
# date features
print('Date features...')
# init
cls_transformer = pre.DateFeatures(
    bool_verbose=True, 
    str_message='Date features',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# dict for rounding
print('Rounding income, loan amount, and vehicle value...')
dict_round = {
    'fltamountfinanced__app': 500,
    'fltapprovedpricewholesale__app': 500,
    'fltgrossmonthly__income_sum': 500,
}
cls_transformer = pre.RoundBinning(
    dict_round=dict_round, 
    str_message='Round income (for PTI), amount financed and vehicle values for (LTV)',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# feature engineering
print('Engineering PTI and LTV...')
# init
cls_transformer = pre.FeatureEngineering(
    bool_verbose=True, 
    str_message='Engineer PTI and LTV',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

######################################################################
# replace inf
print('Replacing inf and -inf...')
# init
cls_transformer = pre.ReplaceInf(
    bool_verbose=True, 
    str_message='Replace inf and -inf with NaN',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# impute with 0 again
print('Imputing numeric with 0...')
# init
cls_transformer = pre.Imputer(
    bool_verbose=True, 
    str_message='Imputer',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

######################################################################
# map term
print('Mapping term...')
cls_transformer = pre.MapTerm(
    str_message='Map term',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# map term
print('Mapping PTI...')
cls_transformer = pre.MapPTI(
    str_message='Map PTI',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

# binning
print('Binning...')
dict_round = {
    'fltamountfinanced__app': 500,
    'fltapproveddowntotal__app': 500,
    'fltapprovedpricewholesale__app': 500,
    'fltapprovedservicecontract__app': 500,
    'fltdowncash__app': 500,
    'fltgapinsurance__app': 500,
    'intservicecontractmileage__app': 10000,
}
cls_transformer = pre.RoundBinning(
    dict_round=dict_round, 
    str_message='Round values',
)
df = cls_transformer.fit_transform(df)
list_transformers.append(cls_transformer)
show_transformers(list_transformers)

######################################################################
# show NaN
print('Showing NaN columns...')
ser_isnull = df.isnull().sum()
ser_isnull = ser_isnull[ser_isnull > 0]
print(ser_isnull)

# check for missing
print('Checking for NaN...')
int_sum_na = df.isnull().sum().sum()
# logic
if int_sum_na > 0:
    raise DataContainsNaN('The data contains missing observations.')
else:
    print(f'There are {int_sum_na} missing obervations')

#####################################################################################
# INITIALIZE PREPROCESSING MODEL
#####################################################################################
print('Creating preprocessing model...')
cls_model_preprocessing = pre.PreprocessingModel(
    list_transformers=list_transformers,
)

#####################################################################################
# SAVE PREPROCESSING MODEL
#####################################################################################
print('Saving preprocessing model...')
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_model_preprocessing, open(str_local_path, 'wb'))
# upload
str_bucket_path = f'01_ad/02_model/00_preprocessing/01_create_preprocessor/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
os.remove(str_local_path)

#####################################################################################
# SAVE PREPROCESSING SCRIPT
#####################################################################################
print('Saving preprocessing script...')
str_filename = 'preprocessing.py'
str_local_path = f'./{str_filename}'
str_bucket_path = f'01_ad/02_model/00_preprocessing/01_create_preprocessor/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
os.remove(str_local_path)

Writing script.py


### Build and push to ECR

In [6]:
%%sh

# name the image
image=genxii-ad-preprocessing

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  134.1kB
Step 1/8 : FROM python:3.9
 ---> 0b5d9d43627d
Step 2/8 : RUN apt-get update
 ---> Using cache
 ---> 641d17ccbb6f
Step 3/8 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 328f7967d740
Step 4/8 : COPY requirements.txt .
 ---> Using cache
 ---> 7a06a226c80b
Step 5/8 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> 289a5b866f3b
Step 6/8 : COPY preprocessing.py .
 ---> Using cache
 ---> a980f685d49f
Step 7/8 : COPY script.py .
 ---> 14ab5b8f692c
Step 8/8 : CMD ["python3", "script.py"]
 ---> Running in 645fffcc4de4
Removing intermediate container 645fffcc4de4
 ---> 0a6fd8989c70
Successfully built 0a6fd8989c70
Successfully tagged genxii-ad-preprocessing:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-ad-preprocessing' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-preprocessing]
e1e2788f5578: Preparing
878acdbed672: Preparing
7fe52816dc80: Preparing
3d6231334b12: Preparing
4ed8de3417bb: Preparing
6d890725d1de: Preparing
e86ea8fbd7dc: Preparing
70981c1da3c1: Preparing
6a4ba3269682: Preparing
d3de4ba9f72c: Preparing
0c2d6fc19d6a: Preparing
2ef3351afa6d: Preparing
5cc3a4df1251: Preparing
2fa37f2ee66e: Preparing
6d890725d1de: Waiting
e86ea8fbd7dc: Waiting
70981c1da3c1: Waiting
6a4ba3269682: Waiting
d3de4ba9f72c: Waiting
0c2d6fc19d6a: Waiting
2ef3351afa6d: Waiting
5cc3a4df1251: Waiting
2fa37f2ee66e: Waiting
3d6231334b12: Layer already exists
7fe52816dc80: Layer already exists
4ed8de3417bb: Layer already exists
878acdbed672: Layer already exists
6d890725d1de: Layer already exists
e86ea8fbd7dc: Layer already exists
70981c1da3c1: Layer already exists
6a4ba3269682: Layer already exists
d3de4ba9f72c: Layer already exists
2ef3351afa6d: Layer already exists
0c2d6fc19d6a: L

### Clean-up

In [7]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py','preprocessing.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass